# 必要リソースの推定

この章では、`profile` サブコマンドで pipeline state(JSON) を解析し、必要リソースを見積もります。
比較対象は前章で生成した `Dim2` / `Dim3` / `DistributedDim2` / `PBC` の 4 ケースです。


この章で確認する内容は次のとおりです。

- `profile` は pipeline state JSON を入力に取ること
- `runtime`, `gate_count`, `code_distance`, `num_physical_qubits` を横比較すること
- PBC モードで値がどう変化するかを読むこと


In [ ]:
import json
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## 0. profile コマンド確認

In [ ]:
!qret profile --help

なお、`asm` で出力したファイルについては `profile` はできません。
`profile` は pipeline state の `parameter` / `opt` / `metadata` を使って集計します。

## 1. 4 ケースの pipeline state を生成

In [ ]:
dim2_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_pipeline.yaml"
dim3_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_pipeline.yaml"
dist_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_pipeline.yaml"
pbc_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc_pipeline.yaml"

!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret compile --verbose --pipeline {pbc_pipeline_path}

## 2. profile を実行

In [ ]:
!qret profile -i {output_dir / "tutorial_5_dim2.json"} -o {output_dir / "tutorial_6_dim2_profile.json"}
!qret profile -i {output_dir / "tutorial_5_dim3.json"} -o {output_dir / "tutorial_6_dim3_profile.json"}
!qret profile -i {output_dir / "tutorial_5_dist.json"} -o {output_dir / "tutorial_6_dist_profile.json"}
!qret profile -i {output_dir / "tutorial_5_pbc.json"} -o {output_dir / "tutorial_6_pbc_profile.json"}

## 3. 主要指標を表で比較

In [ ]:
def summarize(path: str) -> dict:
    data = json.loads(pathlib.Path(path).read_text())
    return {
        "execution_time_sec": data.get("execution_time_sec"),
        "gate_count": data.get("gate_count"),
        "magic_state_consumption_count": data.get("magic_state_consumption_count"),
        "code_distance": data.get("code_distance"),
        "physical_qubit_count": data.get("physical_qubit_count"),
    }


for name, path in [
    ("Dim2", output_dir / "tutorial_6_dim2_profile.json"),
    ("Dim3", output_dir / "tutorial_6_dim3_profile.json"),
    ("DistributedDim2", output_dir / "tutorial_6_dist_profile.json"),
    ("PBC", output_dir / "tutorial_6_pbc_profile.json"),
]:
    print(f"[{name}]")
    for k, v in summarize(path).items():
        print(f"  {k}: {v}")
    print()

## 4. PBC の結果 JSON を確認

In [ ]:
Code(filename=output_dir / "tutorial_6_pbc_profile.json", language="json")

## visualize_compile_info で比較する
`profile_*.json` を GUI で比較したい場合は、`visualize_profile.py` を起動します。

```sh
streamlit run ../../../../quration-visualizer/visualize_profile.py
```

主なタブ:
- `Overview`: 主要指標の比較、`Baseline` 指定で差分確認
- `Tables`: 詳細テーブル（`gate_count_detail` が無い JSON では Details は空）
- `Time Series`: シリーズ選択と beat 範囲指定
- `Topology`: トポロジー footprint 比較
